<h2>Código do projeto de equivalências normativas</h2>

In [2]:
"""

Bibliotecas utilizadas para a raspagem de dados (mais informações estão disponíveis no READ.md)

"""

import requests
from bs4 import BeautifulSoup
from datetime import datetime, date, time
import dateparser
import re
import json
import os
from urllib.parse import unquote
import copy


<h3>Abertura de resoluções em arquivo</h3>

In [2]:
"""

As resoluções da Anatel presentes na pasta legal são abertas utilizando a biblioteca "os" e seus path's são armazenados em um dicionário aninhado.
Esse dicionário funciona de forma que duas chaves devem ser usadas para acessar informações: o ano e o número da resolução. 
Por exemplo "[1997][4]" corresponderá a resolução número 4 de 1997.

"""

pasta_ano = "legal/resolucoes_anatel"
resolucoes_por_ano = {}

for ano in os.listdir(pasta_ano):
    caminho_ano = os.path.join(pasta_ano, ano)
    resolucoes_por_ano[ano] = {}

    for arquivo in os.listdir(caminho_ano):
        caminho_html = os.path.join(caminho_ano, arquivo)
        match = re.search(r"resolu[cç][aã]o(?:\s+anatel)?\s+.*?n[º°]\s*(\d+)",arquivo,re.IGNORECASE)
        if match:
            numero = match.group(1)
            resolucoes_por_ano[ano][numero] = caminho_html

In [3]:
print(resolucoes_por_ano['2025'])
print('')
print(resolucoes_por_ano['1999']['166'])

{'772': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel  nº  772, de 16 de janeiro de 2025.html', '773': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel nº 773, de 16 de janeiro de 2025.html', '774': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel nº 774, de 19 de fevereiro de 2025.html', '775': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel nº 775, de 28 de abril de 2025.html', '776': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel nº 776, de 28 de abril de 2025.html', '777': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel nº 777, de 28 de abril de 2025.html', '778': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel nº 778, de 28 de abril de 2025.html', '779': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel nº 779, de 28 de abril de 2025.html', '780': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Anatel nº 780, de 1º de agosto de 2025.html', '781': 'legal/resolucoes_anatel\\2025\\Anatel - Resolução Ana

<h3>Criação do grafo de equivalências normativas</h3>

In [4]:
"""

Classe responável por extrair e armazenas os metadados de cada resolução.

"""

class Node:
    
    def __init__(self,caminho):
        
        self.soup = self.abrir_arquivo(caminho)
            
        self.title = self.soup.find("h1", class_="documentFirstHeading").get_text(" ", strip=True)
            
        self.data= self.get_published(self.soup)
            
        self.revoked= self.revogation()

        self.next= self.proxima()

        if self.next is not None:
            self.ano_proxima, self.numero_proxima = self.info_proxima(self.next)
        else:
            self.ano_proxima = None
            self.numero_proxima = None

    def abrir_arquivo(self,caminho):
        with open(caminho, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "html.parser")
            return soup

    def get_published(self,soup):
        """

        Retorna a data de publicação da resolução a partir da classe em HTML "documentPublished" no formato datetime.
    
        """
        data=soup.find(class_="documentPublished")
        data=data.get_text(strip=True)
        data=data[11:]
        return dateparser.parse(data)

    def revogation(self):
        """

        Retorna um booleano para indicar se a resolução foi revogada buscando no próprio título por essa informação.
    
        """
        if "(REVOGADA)" in self.title:
            return "Revogada"
        else:
            return "Em vigor"

    def extrair_ano_numero(self,href: str):
        """
        
        Extrai 'ano/numero' de uma URL de resolução, e retorna None caso não consiga.
        
        """
        if not href:
            return None
            
        href = unquote(href.split("#")[0])
    
        #Extrair ano da resolução revogadora
        match_ano = re.search(r"/resolucoes/([^/]+)/", href, re.IGNORECASE)
        
        if not match_ano:
            return None

        ano_bruto = match_ano.group(1)
    
        if re.match(r"\d{4}", ano_bruto): #Caso ano = /2023/
            ano = ano_bruto[:4]
        elif "-" in ano_bruto: #Caso ano = /20-2005/
            ano = ano_bruto.split("-")[-1]
        else:
            return None
    
        #Extrair número da resolução revogadora
        match_num = re.search(r"resolu[cç][aã]o[^0-9]*?(\d+)", href, re.IGNORECASE)
        if not match_num:
            return None
    
        numero = match_num.group(1)
    
        return f"{ano}/{numero}"
            
    def proxima(self):
        """
        
        Retorna a próxima resolução (revogadora) no formato 'ano/numero'ou None se não houver. Composta de três etapas para englobar
        todos os casos.
        
        """
        # Verificar se no título há a palavra "Revogada"
        titulo = self.soup.find("title").get_text().lower()
        if "revogada" not in titulo:
            return None
    
        # Buscar elementos com Revogado(a) pela
        elementos = self.soup.find_all(lambda tag: tag.name in ["td", "p", "a"] and tag.get_text() and
            "vide" not in tag.get_text().lower() and any(keyword in tag.get_text().lower()
            for keyword in ["revogada pela", "revogado pela"]))
    
        for elem in elementos:
            if elem.name == "a":
                link = elem
            else:
                links = elem.find_all("a")
                for link in links:
                    if "revogada pela" in link.get_text().lower() or "revogado pela" in link.get_text().lower():
                        break
            if not link:
                continue

            resultado = self.extrair_ano_numero(link.get("href"))
            if resultado:
                return resultado
    
        # Procurar links com "Resolução nº", que não tenham vide e tenham "revogad" de alguma forma
        for link in self.soup.find_all("a", href=True):
    
            texto_link = link.get_text(" ", strip=True).lower()
    
            if "vide" in texto_link:
                continue
    
            if "resolução nº" not in texto_link:
                continue
    
            contexto = link.parent.get_text(" ", strip=True).lower()
    
            if "vide" in contexto:
                continue
    
            if "revogad" not in contexto:
                continue
    
            resultado = self.extrair_ano_numero(link["href"])
            if resultado:
                return resultado
    
        return None
    
    def info_proxima(self,href):
        partes = href.split("/")          
        ano = partes[-2]                 
        numero = partes[-1].split("-")[-1]
        return ano,numero
        
        

In [5]:
"""

Teste de próximas resoluções.

"""
testando=Node(resolucoes_por_ano['2014']['632'])
print(testando.title)
print(testando.next)
print(testando.ano_proxima)
print(testando.numero_proxima)

Resolução nº 632, de 7 de março de 2014 (REVOGADA)
2023/765
2023
765


In [23]:
"""

Teste se todas as resoluções que possuem Revogada no título possuem uma próxima.

"""
for ano in sorted(resolucoes_por_ano.keys(), key=int):
    for numero in sorted(resolucoes_por_ano[ano].keys(), key=int):

        caminho = resolucoes_por_ano[ano][numero]

        node = Node(caminho)

        titulo = node.title.lower()

        tem_revogada_no_titulo = "revogada" in titulo

        proxima = node.proxima()  # ← usa sua função robusta

        if tem_revogada_no_titulo and not proxima:
            print(f"{ano}/{numero}")

In [6]:
"""

Armazena todas as resoluções no formato 'ano'/'numero' no dicionário nodes.

"""

nodes = {}

# 🔹 Ordenar anos de forma crescente
for ano in sorted(resolucoes_por_ano.keys(), key=int):
    resolucoes = resolucoes_por_ano[ano]
    
    # 🔹 Ordenar resoluções de cada ano por número crescente
    for numero in sorted(resolucoes.keys(), key=int):
        caminho = resolucoes[numero]
        chave = f"{ano}/{numero}"
        
        if chave not in nodes:
            nodes[chave] = Node(caminho)

In [7]:
"""

Armazena dentro do dicionário todas as arestas do grafo.

"""
arestas = {}
visitados = set()
        
for ano, resolucoes in resolucoes_por_ano.items():
    for numero in sorted(resolucoes.keys(), key=int):

        chave_inicial = f"{ano}/{numero}"

        if chave_inicial in visitados:
            continue

        atual = Node(resolucoes[numero])
        atual_chave = chave_inicial

        while atual.next is not None:
            
            if atual_chave in visitados:
                break
                
            visitados.add(atual_chave)
            prox_ano = atual.ano_proxima
            prox_numero = atual.numero_proxima
            prox_chave = f"{prox_ano}/{prox_numero}"

            if (prox_ano in resolucoes_por_ano and prox_numero in resolucoes_por_ano[prox_ano]):

                if atual_chave not in arestas:
                    arestas[atual_chave] = []

                if prox_chave not in arestas:
                    arestas[prox_chave] = []

                arestas[atual_chave].append(prox_chave)
                atual = Node(resolucoes_por_ano[prox_ano][prox_numero])
                atual_chave = prox_chave
            else:
                break

print(arestas)

{'1997/1': ['1999/197'], '1999/197': ['2001/270'], '2001/270': ['2013/612'], '2013/612': [], '1997/2': ['2019/708'], '2019/708': [], '1997/3': ['2019/708'], '1997/4': ['2019/708'], '1998/5': ['2019/708'], '1998/6': ['2019/708'], '1998/7': ['2019/708'], '1998/8': ['2019/708'], '1998/9': ['2019/708'], '1998/10': ['2019/708'], '1998/11': ['2019/708'], '1998/12': ['2019/708'], '1998/13': ['2019/708'], '1998/14': ['2019/708'], '1998/15': ['2019/708'], '1998/16': ['2019/708'], '1998/17': ['2019/708'], '1998/18': ['2019/708'], '1998/19': ['2019/708'], '1998/20': ['2019/708'], '1998/21': ['2019/708'], '1998/22': ['2019/708'], '1998/23': ['2019/708'], '1998/24': ['2019/708'], '1998/25': ['2019/708'], '1998/26': ['2019/708'], '1998/27': ['2019/708'], '1998/28': ['2019/708'], '1998/29': ['2019/708'], '1998/30': ['2019/708'], '1998/31': ['2022/752'], '2022/752': [], '1998/32': ['2019/708'], '1998/33': ['2007/458'], '2007/458': ['2019/708'], '1998/34': ['2019/708'], '1998/35': ['2019/708'], '1998/3

In [20]:
""" 

Criação do Json que armazena os grafos. Os nós contém os metadados e as arestas possuem a resolução de origem e por qual ela foi revogada.

"""

nodes_json = []
edges_json = []

#Nós
for chave, node in nodes.items():
    ano, numero = chave.split("/")
    nodes_json.append({
        "Resolução": chave,
        "Número": f"{numero}",
        "title": node.title,
        "Vigência":node.data.isoformat(),
        "Status":node.revoked,
        "group": ano   # 👈 isso separa por ano no frontend
    })

#Arestas
for origem, destinos in arestas.items():
    for destino in destinos:
        edges_json.append({
            "from": origem,
            "to": destino
        })

#Grafo
grafo_json = {
    "nodes": nodes_json,
    "edges": edges_json
}

with open("grafo.json", "w", encoding="utf-8") as f:
    json.dump(grafo_json, f, ensure_ascii=False, indent=4)


<h3> Grafo com equivalências entre dispositivos</h3>

In [3]:

pasta_dispositivos = "grafos_dispositivos"

# Grafo completo
grafo_equivalencias_dispositivos = {
    "nodes": [],
    "edges": []
}

for arquivo in os.listdir(pasta_dispositivos):
    if arquivo.endswith(".json"):
        caminho_arquivo = os.path.join(pasta_dispositivos, arquivo)
        
        with open(caminho_arquivo, "r", encoding="utf-8") as f:
            dados = json.load(f)
            for node in dados.get("nodes", []):
                partes = node.split("_")
                novo_node = {
                    "id": node,
                    "Resolução": partes[0],
                    "dispositivo": partes[1] 
                }

                grafo_equivalencias_dispositivos["nodes"].append(novo_node)
            grafo_equivalencias_dispositivos["edges"].extend(dados.get("edges", []))

with open("grafo_equivalencias_dispositivos.json", "w", encoding="utf-8") as f:
    json.dump(grafo_equivalencias_dispositivos, f, indent=4, ensure_ascii=False)

<h3>Raspagem de dados com URL's</h3>

In [ ]:
"""

Sessão para testar a abertura de url's diferentes 

"""

url = "https://informacoes.anatel.gov.br/legislacao/resolucoes/2000/563-resolucao-252"
url2 = "https://informacoes.anatel.gov.br/legislacao/leis/473-lei-9295"
url3 = "https://informacoes.anatel.gov.br/legislacao/resolucoes/2012/191-resolucao-589"
resposta = requests.get(url)

print(resposta.status_code)

print(resposta.text[:100])

In [14]:
class Documento:
    """
    
    Classe principal Documento, ela abrange todas as funções que serão necessárias para a raspagem de dados.Além disso, a classe
    possui um construtor capaz de extrair informações de uma url bem como extrair as mesmas informações a partir de um JSON passado.
    
    """
    def __init__(self,resolucao):
        """
    
        Como em python não é possível fazer sobrecarga de métodos propriamente, o construtor verifica se a resolução passada é uma url por meio do uso
        de expressões regulares, caso contrário a classe entende que a resolução está no formato JSON.
    
        """
        if self.is_url(resolucao):
            resposta = requests.get(resolucao)
            soup = BeautifulSoup(resposta.text, "html.parser")
            self.title=soup.title.string
            self.data= self.get_published(soup)
            self.revoked= self.revogation()
            self.text=self.get_text(soup)
        else:      
            with open(resolucao, "r", encoding="utf-8") as f:
                json_doc=json.load(f)      
            self.title= json_doc[0]["Título"]
            self.data= datetime.fromisoformat(json_doc[0]["Data de Publicação"])
            self.revoked= json_doc[0]["Revogada"]
            self.text= json_doc[0]["Texto"]


    def is_url(self, resolucao):
        """
    
        Utiliza expressões regulares para retornar um booleano que indica que se a resolução é uma url.
    
        """
        if re.match(r"https?://", resolucao):
            return True
        return False

    def get_published(self,soup):
        """

        Retorna a data de publicação da resolução a partir da classe em HTML "documentPublished" no formato datetime.
    
        """
        data=soup.find(class_="documentPublished")
        data=data.get_text(strip=True)
        data=data[11:]
        return dateparser.parse(data)

    def revogation(self):
        """

        Retorna um booleano para indicar se a resolução foi revogada buscando no próprio título por essa informação
    
        """
        if "(REVOGADA)" in self.title:
            return True
        else:
            return False

    def redacao(self,i,p,paragrafos):
        """

        Retorna a informação se a linha atual sofreu uma nova redação(Ainda está sendo propriamente desenvolvida)

        """
        texto = p.get_text(strip=True)
        prox=i+1
        if prox>=len(paragrafos):
            return False
        texto_proximo= paragrafos[prox].get_text(strip=True)
        redacao_regex = re.compile(r"\(Redação.*")
        style = p.get("style", "")   
        if "line-through" in style:
            redacao_match = redacao_regex.search(texto_proximo) 
            return True
        else:
            return False

    def get_itens(self,documento,i,linhas,artigo_atual):
        """

        Cria um loop que irá analisar todos os itens dentro de um artigo e juntá-los ao artigo correspondente 
        (deve abranger ainda letras mas está em desenvolvimento). Além disso, a função conta quantas linhas 
        foram usadas para que elas possam ser puladas.

        """
        sentinel_itens=False
        contador_itens=0
        while sentinel_itens == False:
            if i+contador_itens +1 >= len(linhas):
                texto_atual=linhas[i+contador_itens].get_text(strip=True)
                texto_atual = texto_atual.replace(u'\u00A0', ' ')
                documento[artigo_atual]["itens"].append(texto_atual)
                break
            else:
                texto_atual=linhas[i+contador_itens].get_text(strip=True)
                texto_atual = texto_atual.replace(u'\u00A0', ' ')
                documento[artigo_atual]["itens"].append(texto_atual)
                texto_proximo = linhas[i+1+contador_itens].get_text(strip=True)
                texto_proximo = texto_proximo.replace(u'\u00A0', ' ')
                if re.match(r"Art\.\s*\d+(?:-[A-Za-z])?", texto_proximo) or re.match(r"§\s*\d+", texto_proximo):
                    sentinel_itens=True
                else:
                    contador_itens+=1
        skip=contador_itens
        return skip

    def get_paragrafos(self,documento,i,linhas,artigo_atual):
        """

        Cria um loop que irá analisar todos os parágrafos dentro de um artigo, bem como os itens desses parágrafos e juntá-los
        ao artigo correspondente (deve abranger ainda letras mas está em desenvolvimento).Além disso, a função conta quantas 
        linhas foram usadas para que elas possam ser puladas.

        """
        sentinel_paragrafos=False
        contador_paragrafos=0
        while sentinel_paragrafos == False:
            if i+contador_paragrafos >= len(linhas):
                texto_atual=linhas[i+contador_paragrafos].get_text(strip=True)
                texto_atual = texto_atual.replace(u'\u00A0', ' ')
                documento[artigo_atual]["paragrafos"].append(texto_atual)
                break
            else:
                texto_atual=linhas[i+contador_paragrafos].get_text(strip=True)
                texto_atual = texto_atual.replace(u'\u00A0', ' ')
                documento[artigo_atual]["paragrafos"].append(texto_atual)
                texto_proximo = linhas[i+1+contador_paragrafos].get_text(strip=True)
                texto_proximo = texto_proximo.replace(u'\u00A0', ' ')
                if re.match(r"Art\.\s*\d+(?:-[A-Za-z])?", texto_proximo):
                    sentinel_paragrafos=True
                else:
                    contador_paragrafos+=1
        skip=contador_paragrafos
        return skip

    def get_text(self,soup):
        """

        Loop principal que coleta todo o texto da url e o armazena em um dicionário cuja chave é o número do artigo.Além disso,
        caso tenham anexos na resolução a função adiciona "AnexoX" ao nome da chave do artigo correspondente.

        """
        linhas = soup.find_all("p", class_="texto-recuo-1a-linha")
        documento = {}
        artigo_atual = None
        contador=0
        skip=0

        for i,p in enumerate(linhas):
            if skip == 0:
                texto = p.get_text(strip=True)
                texto = texto.replace(u'\u00A0', ' ')
                new_artigo= re.match(r"Art\.\s*\d+(?:-[A-Za-z])?", texto)
                if self.redacao(i,p,linhas):
                        continue
                else:
                    if new_artigo:
                        artigo_atual = new_artigo.group()
                        artigo_atual= artigo_atual.replace(" ","")
                        if artigo_atual not in documento and contador==0:
                            documento[artigo_atual] = {"texto": texto, "itens": [],"paragrafos":[]}
                        elif artigo_atual in documento and artigo_atual == "Art.1" :
                            contador+=1
                            artigo_atual=artigo_atual+"Anexo"+str(contador)
                            documento[artigo_atual] = {"texto": texto, "itens": [],"paragrafos":[]}
                        else:
                            artigo_atual=artigo_atual+"Anexo"+str(contador)
                            documento[artigo_atual] = {"texto": texto, "itens": [],"paragrafos":[]}

                    paragraph_match= re.match(r"§\s*\d+", texto)
                    item_match = re.match(r"[IVXL]", texto)
                    if item_match:
                        skip=self.get_itens(documento,i,linhas,artigo_atual)
            
                    if paragraph_match:
                        skip=self.get_paragrafos(documento,i,linhas,artigo_atual)
 
            else:
                skip-=1        
        return documento    

    def create_json(self):
        """

        Cria o JSON a partir dos metadados de uma resolução separando o JSON em: "Título","Data de Publicação","Revogada","Texto".
        O nome do JSON criado é o mesmo nome do título da resolução.

        """
        data_str = self.data.isoformat()
        info=[]
        info.append({"Título": self.title,"Data de Publicação": data_str,"Revogada": self.revoked,"Texto":self.text})
        json_doc = json.dumps(info, ensure_ascii=False, indent=4)
        with open(f"{self.title}.json", "w", encoding="utf-8") as f:
            json.dump(info, f, ensure_ascii=False, indent=4)
        

In [15]:
"""

Sessão de testes de criação do JSON a partir de uma url.

"""
teste=Documento("https://informacoes.anatel.gov.br/legislacao/resolucoes/2007/264-resolucao-466")
print(teste.title)
#print(teste.data)
#print(teste.revoked)
#print(teste.text['Art.1'])
#teste.create_json()

#teste_2=Documento("https://informacoes.anatel.gov.br/legislacao/resolucoes/2000/563-resolucao-252")
#print(teste_2.title)
#print(teste_2.data)
#print(teste_2.revoked)
#print(teste_2.text)
#teste_2.create_json()

Anatel - Resolução nº 466, de 16 de maio de 2007 (REVOGADA)


In [ ]:
"""

Sessão de teste da transformação de um JSON nos elementos da classe novamente.

"""
teste_3=Documento("Anatel - Lei nº 9.295, de 19 de julho de 1996.json")
print(teste_3.title)
print(teste_3.data)
print(teste_3.revoked)
print(teste_3.text)